<a href="https://colab.research.google.com/github/Traffic-Forecasting-Thesis-Group/Event-Aware-Traffic-Forecasting/blob/feature%2Fmodule-1-and-2/fuse_traffic_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fuse-Traffic: Transformer-Based Binary Label Training

This notebook trains the Metro Manila traffic signal classifiers on the labeled master dataset:
1) Pre-clean text
2) DistilBERT (binary classification)
3) RoBERTa (binary classification)
4) Ensemble decision logic + metrics

**Note:** This notebook now loads the completed labeled master file at `ml/data/x_labeled_master.csv`.

## 1. Dependencies & Setup

In [ ]:
!pip -q install transformers datasets torch sentencepiece sacremoses evaluate accelerate scikit-learn

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 2. Data Loading

In [ ]:
import os
from pathlib import Path

import pandas as pd

# If running in Colab, mount Drive so the labeled master file becomes available.
colab_mode = False
try:
    from google.colab import drive, files  # type: ignore
    colab_mode = True
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
except Exception:
    files = None

# Load the completed labeled master dataset.
# Priority order:
# 1) explicit LABELED_PATH environment variable
# 2) notebook-local repo paths
# 3) the nested notebook-relative path used by the attachment
# 4) Colab Drive path
# 5) interactive upload in Colab
candidate_paths = []
explicit_path = os.getenv('LABELED_PATH', '').strip()
if explicit_path:
    candidate_paths.append(Path(explicit_path))

candidate_paths.extend([
    Path('./ml/data/x_labeled_master.csv'),
    Path('./ml/notebooks/ml/data/x_labeled_master.csv'),
    Path('./x_labeled_master.csv'),
    Path('../data/x_labeled_master.csv'),
    Path('../../ml/data/x_labeled_master.csv'),
    Path('/content/x_labeled_master.csv'),
    Path('/content/drive/MyDrive/x_labeled_master.csv'),
])

LABELED_PATH = None
for path in candidate_paths:
    if path.exists():
        LABELED_PATH = path
        break

if LABELED_PATH is None and colab_mode and files is not None:
    print('x_labeled_master.csv was not found. Upload it now from your computer.')
    uploaded = files.upload()
    for uploaded_name in uploaded.keys():
        uploaded_path = Path('/content') / uploaded_name
        if uploaded_path.exists() and uploaded_name.lower().endswith('.csv'):
            LABELED_PATH = uploaded_path
            break

if LABELED_PATH is None:
    raise FileNotFoundError(
        'Could not find x_labeled_master.csv. Set LABELED_PATH or place the file in ml/data, ml/notebooks/ml/data, /content, or /content/drive/MyDrive.'
    )

print(f'Using labeled dataset: {LABELED_PATH}')

x_labeled_master.csv was not found. Upload it now from your computer.


In [3]:
df = pd.read_csv(LABELED_PATH)

# Normalize column names and required fields.
if 'x_post_id' in df.columns and 'post_id' not in df.columns:
    df = df.rename(columns={'x_post_id': 'post_id'})

required_cols = ['post_id', 'created_at', 'raw_text', 'translated_text', 'reliability_label']
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f'Missing required column: {col}')

if 'source_type' not in df.columns:
    df['source_type'] = 'unknown'

if 'translated_text' not in df.columns or df['translated_text'].isna().all():
    df['translated_text'] = df['raw_text']

print(f'Loaded {len(df)} labeled rows')
print('Source distribution:')
print(df['source_type'].value_counts(dropna=False))
print('\nLabel distribution:')
print(df['reliability_label'].value_counts(dropna=False).sort_index())
df.head()

NameError: name 'pd' is not defined

### Binary Training Data
This notebook uses the existing `reliability_label` column as the binary target for both models.

In [ ]:
LABEL_STUB_PATH = Path('./ml/data/x_labeled_master.csv')

if 'raw_text' not in df.columns:
    for candidate in ['text', 'content', 'translated_text']:
        if candidate in df.columns:
            df['raw_text'] = df[candidate]
            break

if 'translated_text' not in df.columns:
    df['translated_text'] = df['raw_text']

if 'raw_text' not in df.columns:
    raise ValueError('No text column found. Add one of: raw_text, text, content, translated_text.')

selected_cols = [
    c for c in ['post_id', 'created_at', 'raw_text', 'translated_text', 'source_type', 'reliability_label']
    if c in df.columns
]

label_stub = df[selected_cols].copy()
print('Using labeled master file directly, not a manual stub.')
print(f'Rows available for training: {len(label_stub)}')
label_stub.head()

After loading the labeled master file, continue to preprocessing and model training.

In [ ]:
labeled_df = pd.read_csv(LABELED_PATH)
print(f'Loaded labeled dataset: {len(labeled_df)} rows')
print(labeled_df[['post_id', 'reliability_label']].head())

## 3. Text Preprocessing

In [4]:
import re

class TextPreprocessor:
    def clean(self, text):
        text = re.sub(r'http\S+|www\.\S+', '', str(text))
        text = re.sub(r'@\w+', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

pre = TextPreprocessor()

if 'translated_text' not in labeled_df.columns:
    labeled_df['translated_text'] = labeled_df['raw_text']

labeled_df['clean_text'] = labeled_df['translated_text'].fillna(labeled_df['raw_text']).apply(pre.clean)
labeled_df.head()

NameError: name 'labeled_df' is not defined

## 4. DistilBERT Fine-Tuning (Binary Classification)

In [ ]:
from datasets import Dataset
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments
import evaluate

binary_df = labeled_df.dropna(subset=['clean_text', 'reliability_label']).copy()
binary_df['reliability_label'] = pd.to_numeric(binary_df['reliability_label'], errors='coerce')
binary_df = binary_df.dropna(subset=['reliability_label'])
binary_df['reliability_label'] = binary_df['reliability_label'].astype(int)

if len(binary_df) < 20:
    raise ValueError('Not enough labeled rows for DistilBERT training. Add more reliability_label annotations.')

# Keep binary labels only.
if set(binary_df['reliability_label'].unique()) - {0, 1}:
    raise ValueError('reliability_label must be binary (0 or 1) for DistilBERT training.')

dataset = Dataset.from_pandas(binary_df[['clean_text', 'reliability_label']])
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def tokenize_batch(batch):
    return tokenizer(batch['clean_text'], truncation=True, padding='max_length', max_length=128)

dataset = dataset.map(tokenize_batch, batched=True)
dataset = dataset.rename_column('reliability_label', 'labels')
dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

split = dataset.train_test_split(test_size=0.2, seed=42)
accuracy = evaluate.load('accuracy')
f1 = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {
        'accuracy': accuracy.compute(predictions=preds, references=labels)['accuracy'],
        'f1': f1.compute(predictions=preds, references=labels)['f1']
    }

model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

args_kwargs = dict(
    output_dir='./ml/models/distilbert_binary',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    report_to='none',
)
if 'eval_strategy' in TrainingArguments.__init__.__code__.co_varnames:
    args_kwargs['eval_strategy'] = 'epoch'
else:
    args_kwargs['evaluation_strategy'] = 'epoch'
args = TrainingArguments(**args_kwargs)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=split['train'],
    eval_dataset=split['test'],
    compute_metrics=compute_metrics,
)

trainer.train()
trainer.evaluate()

NameError: name 'labeled_df' is not defined

## 5. RoBERTa Fine-Tuning (Binary Classification)

In [ ]:
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    RobertaTokenizerFast,
    RobertaForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.metrics import accuracy_score
import evaluate

multi_df = labeled_df.dropna(subset=['clean_text', 'reliability_label']).copy()
multi_df['reliability_label'] = pd.to_numeric(multi_df['reliability_label'], errors='coerce')
multi_df = multi_df.dropna(subset=['reliability_label'])
multi_df['reliability_label'] = multi_df['reliability_label'].astype(int)

if len(multi_df) < 20:
    raise ValueError('Not enough labeled rows for RoBERTa training. Add more reliability_label annotations.')

if set(multi_df['reliability_label'].unique()) - {0, 1}:
    raise ValueError('reliability_label must be binary (0 or 1) for RoBERTa training.')

num_labels = 2
dataset_m = Dataset.from_pandas(multi_df[['clean_text', 'reliability_label']])
tokenizer_m = RobertaTokenizerFast.from_pretrained('roberta-base')

def tokenize_batch_m(batch):
    return tokenizer_m(batch['clean_text'], truncation=True, padding='max_length', max_length=128)

dataset_m = dataset_m.map(tokenize_batch_m, batched=True)
dataset_m = dataset_m.rename_column('reliability_label', 'labels')
dataset_m.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

split_m = dataset_m.train_test_split(test_size=0.2, seed=42)

counts = np.bincount(multi_df['reliability_label'], minlength=num_labels)
counts = np.where(counts == 0, 1, counts)
weights = counts.sum() / (len(counts) * counts)
class_weights = torch.tensor(weights, dtype=torch.float)

model_m = RobertaForSequenceClassification.from_pretrained('roberta-base', num_labels=num_labels)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get('labels')
        outputs = model(input_ids=inputs['input_ids'], attention_mask=inputs['attention_mask'])
        logits = outputs.get('logits')
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights.to(logits.device))
        loss = loss_fct(logits.view(-1, num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

metric_f1 = evaluate.load('f1')

def compute_metrics_m(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1_macro': metric_f1.compute(predictions=preds, references=labels, average='macro')['f1']
    }

args_m_kwargs = dict(
    output_dir='./ml/models/roberta_binary',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    report_to='none',
)
if 'eval_strategy' in TrainingArguments.__init__.__code__.co_varnames:
    args_m_kwargs['eval_strategy'] = 'epoch'
else:
    args_m_kwargs['evaluation_strategy'] = 'epoch'
args_m = TrainingArguments(**args_m_kwargs)

trainer_m = WeightedTrainer(
    model=model_m,
    args=args_m,
    train_dataset=split_m['train'],
    eval_dataset=split_m['test'],
    compute_metrics=compute_metrics_m
)

trainer_m.train()
trainer_m.evaluate()

## 6. Ensemble Decision Logic

In [ ]:
import numpy as np

DISTIL_THRESHOLD = 0.85
ROBERTA_THRESHOLD = 0.60

def final_decision(distil_positive_prob, roberta_positive_prob):
    if distil_positive_prob >= DISTIL_THRESHOLD:
        return 1
    if roberta_positive_prob >= ROBERTA_THRESHOLD:
        return 1
    return 0

# Placeholder arrays for combined evaluation
# Replace these with real model probabilities after inference.
distil_scores = np.array([0.9, 0.2, 0.7])
roberta_scores = np.array([0.1, 0.8, 0.3])
combined = [final_decision(d, r) for d, r in zip(distil_scores, roberta_scores)]
combined

## 7. Metrics & Export

In [ ]:
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score

# Replace with real predictions and labels after running inference.
y_true = np.array([1, 0, 1])
y_pred = np.array(combined)

precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary')
auc = roc_auc_score(y_true, y_pred)

print('Precision:', precision)
print('Recall:', recall)
print('F1:', f1)
print('AUC:', auc)

model.save_pretrained('./ml/models/distilbert_binary')
tokenizer.save_pretrained('./ml/models/distilbert_binary')
model_m.save_pretrained('./ml/models/roberta_binary')
tokenizer_m.save_pretrained('./ml/models/roberta_binary')

print('Models saved.')